In [216]:
import pandas as pd
import numpy as np

In [217]:
dates = pd.read_csv("../data/external/states_sports_betting_legalization.csv")
state_monthly = pd.read_csv("../data/processed/helpline_monthly.csv")
state_monthly.rename(columns={'caller_state': 'State'}, inplace=True)
dates.rename(columns={'Online Launch': 'online_launch', "Retail Launch": "retail_launch", "Treatment Group": "treatment_group"}, inplace=True)
state_monthly["date"] = pd.to_datetime(state_monthly["date"])
dates["online_launch"] = pd.to_datetime(dates["online_launch"])
dates["retail_launch"] = pd.to_datetime(dates["retail_launch"])

# Panel Spine

In [218]:
all_states = dates[~dates['State'].isin(['Nevada','Florida','Missouri'])]['State'].unique()
all_dates = pd.date_range('2016-01-01', '2025-12-01', freq='MS')
panel = pd.DataFrame([(state, date) for state in all_states for date in all_dates], columns=['State', 'date'])
panel = panel.merge(state_monthly, on=['State', 'date'], how='left')
panel = panel.merge(dates, on='State', how='left')

# Outages and Missing Values

In [219]:
panel[panel["total_contacts"].isna()]["date"].value_counts()

date
2018-08-01    47
2018-12-01    47
2020-04-01    47
2020-08-01    47
2018-03-01     1
2018-04-01     1
2018-07-01     1
2018-11-01     1
Name: count, dtype: int64

In [220]:
# Full outages — entire month missing, already NaN in panel
full_outages = ['2018-08-01', '2018-12-01', '2020-04-01', '2020-08-01']

# Partial outages — data exists but undercounted
partial_outages = ['2016-02-01', '2022-09-01']

panel['telecom_outage'] = panel['date'].astype(str).isin(
    full_outages + partial_outages
).astype(int)

sd_mask = panel['State'] == 'South Dakota'
panel.loc[sd_mask, 'total_contacts'] = panel.loc[sd_mask, 'total_contacts'].interpolate()

# Treatment Variables

In [221]:
panel["treated"] = panel['online_launch'].notna().astype(int)
panel["post"] = (panel['date'] >= panel['online_launch']).astype(int)
panel["treat_post"] = panel["treated"] * panel["post"]
panel["online_only"] = ((panel['online_launch'].notna()) & (panel['retail_launch'].isna())).astype(int)
panel["time_to_treatment"] = panel.apply(
    lambda r: (r['date'].to_period('M') - r['online_launch'].to_period('M')).n 
    if pd.notna(r['online_launch']) else np.nan, axis=1
)

# Population

In [222]:
state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY'
}
panel["State"] = panel["State"].map(state_abbrev)


In [223]:
pop = pd.read_csv("../data/external/historical_state_population_by_year.csv", header=None, names=["State", "year", "Population"])
panel["year"] = panel['date'].dt.year
panel["month"] = panel['date'].dt.month

In [224]:
panel = panel.merge(pop, on=['State', 'year'], how='left')

In [225]:
panel['contacts_per_100k'] = panel['total_contacts'] / panel['Population'] * 100000

In [230]:
# assert len(panel) == 5640
# assert panel['total_contacts'].isna().sum() == 188  # 4 months x 47 states outage months
# assert panel['State'].nunique() == 47